# One value the policy would not say

`08` ended with `flags_acc` as the only thing still binding, and with the five suspects it had
ruled out: not the arithmetic, not domain knowledge, not the trust region, not the baseline,
not gradient variance. This notebook finds what it actually was, and fixes it.

**The finding.** The policy emits `flat` **zero times in 600 flag slots**, and `flat` is the
right answer in 184 of them. Every other value is nearly perfect once the harness supplies
the arithmetic — 399 of 416 non-flat slots, 0.959 — and

$$0.959 \times \frac{416}{600} = 0.665$$

is `flags_acc` to three decimals. The entire remaining deficit of the whole series is one
value the policy will not say.

**It is a prior, not a capability, and the two can be told apart.** Held out by distance to
the threshold, the trained policy calls a non-flat flag correctly on **37 of 38** slots that
sit within 2–5 pp of a boundary. It can do the comparison, and precision near the boundary is
not the problem. `flat` is 0 for 41, 0 for 87 and 0 for 56 across every distance bucket — the
token is not produced on easy cases either.

**The fix is one number.** Paying 3× for a correctly-called `flat` takes held-out
`exact_match` from 0.145 to **0.255** and `flags_acc` from 0.665 to **0.877**, with the
arithmetic still exact at every point and the critic still ahead of a clock. Two other values
of the same knob fail in opposite directions, and a different instrument fails for a reason
worth recording.

## The diagnosis

One greedy pass over the full dev split, with the parsed objects kept, on the frozen policy
and on `08`'s trained one. Three things fall out at once.

### Which flag, and it is not the ordered rule

| | frozen | `08` trained |
| --- | --- | --- |
| `flow` | 0.860 | 0.860 |
| `salt_passage` | 0.455 | 0.700 |
| `dp` | 0.455 | 0.435 |
| all three right | 0.130 | 0.215 |

`salt_passage` is the only four-way flag and the only one whose rules must be read in order
(`sharp_up` at +50 before `up` at +15). RL fixes it on its own: the `sharp_up → up` confusion
falls from 60 cases to 9, and `sharp_up` recall goes 0.302 → 0.895. That suspect is cleared.

### The confusions are one-directional

| true → predicted | frozen | trained |
| --- | --- | --- |
| **`flat` → `up`** | **104** | **121** |
| **`flat` → `down`** | **80** | **63** |
| `sharp_up` → `up` | 60 | 9 |

184 of about 250 errors are a `flat` that was called something else. Nothing is ever called
`flat` by mistake, because nothing is ever called `flat`.

### Capability and prior, separated

Accuracy by distance from the nearest threshold, split on whether the true answer is `flat`:

| `08` trained | 2–5 pp | 5–10 pp | >10 pp |
| --- | --- | --- | --- |
| true value is **not** `flat` | **37/38 = 0.97** | 69/71 = 0.97 | 293/307 = 0.95 |
| true value **is** `flat` | **0/41** | **0/87** | **0/56** |

This is the table the rest of the notebook rests on. A slot 2 pp from its threshold is called
correctly 97% of the time — when the answer is not `flat`. The comparison is not the problem,
and the `flat` failure is flat across every distance, which is what a missing output prior
looks like and not what a precision limit looks like.

**The ceiling this implies.** `07` and `08` measured `cause_given_flags = 1.000` — three
correct flags determine the root cause, and the lookup is then perfect. So if `flat` were
called as accurately as everything else, `flags_acc` would be about 0.96 and `cause_acc`
would follow it up to roughly 0.88, from 0.435.

In [ ]:
import json, sys, collections, statistics as st
from pathlib import Path

HERE = Path.cwd() if (Path.cwd() / "ppo_ac.py").exists() else Path("experiments/notebooks/smoke_test")
sys.path.insert(0, str(HERE.resolve()))
FLAGS = ("flow", "salt_passage", "dp")

def diagnosis(name):
    """Per-slot records from the greedy dev pass: true, predicted, and the
    distance from the case's own value to its nearest threshold."""
    for path in HERE.glob("runs/flag_diagnosis*.json"):
        d = json.load(open(path))
        if name in d:
            return d[name]
    raise KeyError(name)

def split_by_margin(recs, is_flat):
    out = {}
    for lo, hi, lab in ((2, 5, "2-5pp"), (5, 10, "5-10pp"), (10, 1e9, ">10pp")):
        sel = [(r, f) for r in recs for f in FLAGS
               if lo <= r[f]["margin"] < hi and (r[f]["true"] == "flat") == is_flat]
        if sel:
            out[lab] = (sum(r[f]["got"] == r[f]["true"] for r, f in sel), len(sel))
    return out

recs = diagnosis("trained-lam095")
print("08's trained policy, held out on dev, by distance to the nearest threshold")
for label, is_flat in (("true value is NOT flat", False), ("true value IS flat    ", True)):
    cells = split_by_margin(recs, is_flat)
    print(f"  {label}  " + "   ".join(
        f"{k} {h}/{n}={h/n:.2f}" for k, (h, n) in cells.items()))
print()
print("  the comparison is fine; the token is simply never produced")

## Why the prompt cannot fix it

The obvious first move is to tell the model. Four variants, frozen policy, greedy, full dev —
no training, so nothing here is confounded by a moving target:

| variant | `flat` emitted / 184 | `flags_acc` | all three |
| --- | --- | --- | --- |
| v3, as `08` left it | **0** | 0.590 | 0.130 |
| the schema enumerates the allowed values | 1 | 0.633 | 0.210 |
| **the flat band stated first, plus "roughly a third of all flags are flat. Do not avoid it."** | **26** | 0.653 | **0.245** |
| both of the above | 3 | 0.628 | 0.155 |

Told in as many words that a third of the answers are `flat` and not to avoid it, the model
says `flat` 26 times out of 184. That is a fact about the model, not about the wording — and
note the fourth row, where adding the schema enumeration on top *suppresses* `flat` back to 3.

The third variant was promoted to `build_messages_v4` and trained. It regressed:
`flags_acc` 0.653 → 0.605 → 0.558 and stopped at step 68 (`runs/v4-stopped-at-68-regressing`).
0.558 is *below* the 0.665 floor that never saying `flat` produces, so it broke something
else as well — the frozen probe shows what: restating Step 2 over-emits `sharp_up`, 142 times
against 86 true, and RL amplifies that.

**So the prompt is not the lever.** But the probe establishes the thing that decides whether
any lever exists: at temperature 1.0, on the train split, the policy emits `flat` on **3 of
600 slots, 0.50%, and all three were correct**. A policy gradient reweights behaviour that
appears in a sample. 0.5% is thin — about 24 occurrences across a 200-step run — but it is
not zero, and zero is the only number that would have made this unfixable by RL.

## Three instruments, and only one of them is right

### `flat` credit at 8×, which works and costs too much

`field_credits` pays `weights.flags / 3` for each correct flag. Multiplying that by 8 when the
true value is `flat` — and only then, a wrong `flat` still earns nothing:

| | `flat` recall | non-flat, 2–5 pp | non-flat, 5–10 pp | non-flat, >10 pp | `sharp_up` |
| --- | --- | --- | --- | --- | --- |
| `08` trained | 0.000 | 0.97 | 0.97 | 0.95 | 0.895 |
| **8×**, at step 78 | **0.304** | 0.84 | 0.83 | 0.72 | **0.023** |

`flat` recall goes 0.000 → 0.304 — the first time in this series a collapsed output value has
been recovered at all, and it took one coefficient. But non-flat falls in **every** bucket,
uniformly, and `sharp_up` is destroyed to below its frozen value of 0.302. That is not
`flat` and `sharp_up` competing for probability mass; it is an 8× gradient at three tokens
monopolising a trust region that `clip_eps = 0.2` holds fixed. `flags_acc` went 0.590 → 0.552
→ 0.542 and the run was stopped at step 78.

**The giveaway that credit was the wrong frame:** the reward was already neutral. Every flag
pays the same whether or not it is `flat`. A calibrated reward, and the policy still never
says it — so nothing about the *credit* was causing the collapse.

### Targeted entropy, which is the right diagnosis and the wrong tool

An action that is never sampled is an exploration problem, not a credit problem, and the
dense-credit machinery already locates *where* the choice is made. `flag_token_mask` marks
the three tokens that spell the flag values — verified at exactly 3 of 99 tokens on a real
completion — so an entropy bonus can be applied there and nowhere else. `05` measured the
global coefficient blowing completion length up at 0.020; this one touches 3% of the sequence.

It does not work, and the reason is worth more than the run:

| `flag_entropy_coef` | flag-token entropy, first 5 steps → last 5 |
| --- | --- |
| 0.05 | 0.044 → 0.043 (no trend over 20 steps) |
| 0.5 | 0.044 → 0.076 |

At 10× the coefficient the entropy does move, but 0.076 nats against a possible
$\ln 4 = 1.39$ is still a near-deterministic head, and the extrapolation does not reach the
~0.6 that emitting `flat` 30% of the time would need. Pushing it that far is the problem:
**entropy is undirected.** Raising it enough to produce `flat` also randomises the 96% the
policy already gets right — the same trade the 8× credit made, reached along a different
road. Abandoned at step 22, twice.

### `flat` credit at 3×, which is the answer

8× starts the behaviour and monopolises the update; 1× never starts it. 3× is the scale that
does the first without the second.

In [ ]:
def flag_profile(name):
    recs = diagnosis(name)
    hit = collections.Counter(r[f]["true"] for r in recs for f in FLAGS
                              if r[f]["got"] == r[f]["true"])
    tot = collections.Counter(r[f]["true"] for r in recs for f in FLAGS)
    nonflat = (sum(v for k, v in hit.items() if k != "flat"),
               sum(v for k, v in tot.items() if k != "flat"))
    allthree = sum(all(r[f]["got"] == r[f]["true"] for f in FLAGS) for r in recs)
    return {"flat": hit["flat"] / tot["flat"],
            "non_flat": nonflat[0] / nonflat[1],
            "sharp_up": hit["sharp_up"] / tot["sharp_up"],
            "all_three": allthree / len(recs)}

print(f"  {'policy':22}{'flat':>8}{'non-flat':>10}{'sharp_up':>10}{'all three':>11}")
for name in ("frozen", "trained-lam095", "flat8-stopped-at-78", "flat3-final"):
    try:
        p = flag_profile(name)
    except KeyError:
        continue
    print(f"  {name:22}{p['flat']:>8.3f}{p['non_flat']:>10.3f}"
          f"{p['sharp_up']:>10.3f}{p['all_three']:>11.3f}")
print()
print("  frozen and 08-trained never say it; 8x recovers it and breaks everything")
print("  else; 3x is the scale that starts the behaviour without monopolising")

## The result

`ppo-qwen3-17b-v3-flat3-s0` — `flat_credit_scale = 3.0` on top of `08`'s configuration, with
nothing else changed: v3 prompts, dense per-field credit, `lam = 0.95`, `ABLATE`,
`entropy_coef = 0.005`, seed 0, `critic_window = 25`.

| step | `exact_match` | `flags_acc` | `cause_acc` | `08` baseline `exact` |
| --- | --- | --- | --- | --- |
| 0 | 0.090 | 0.590 | 0.295 | 0.090 |
| 100 | 0.165 | 0.678 | 0.305 | 0.150 |
| 150 | 0.190 | **0.808** | 0.405 | 0.135 |
| 175 | 0.240 | 0.860 | 0.385 | 0.155 |
| **200** | **0.255** | **0.877** | 0.420 | 0.145 |

**`exact_match` 0.090 → 0.255**, against 0.145 for the same configuration without this one
coefficient, and against **0.000 at all 27 evaluation points of every run in `runs/` before
`08`**. One completely correct answer in four.

The shape is worth reading. Nothing much happens for 125 steps — `flags_acc` sits at 0.67,
which is the never-says-`flat` floor — and then it moves in one jump between 125 and 150.
That is what a rare action compounding looks like: the policy needs to sample `flat` correctly
often enough to raise its own probability of sampling it, and below some rate that loop does
not close. **And it had not converged at step 200.**

### The three gates, checked at every evaluation point

| | |
| --- | --- |
| the arithmetic | `numeric_acc` **1.000**, minimum over all nine points |
| no regression | points below the 0.665 water line: **0** |
| the critic | residual over a clock **+0.0288**, ahead on **93%** of steps — the baseline's own figures are +0.0319 and 93% |

The critic gate is the one that could have been quietly sacrificed to buy the task result, and
it was not: this run's value function is as far ahead of a token-counter as `08`'s was.

In [ ]:
def evals(run):
    return [json.loads(l) for l in open(HERE / "runs" / run / "eval.jsonl")]

def critic(run, since=25):
    rows = [json.loads(l) for l in open(HERE / "runs" / run / "metrics.jsonl")][since:]
    rows = [r for r in rows if r.get("value_ev") is not None
            and r.get("value_ev_position") is not None]
    ev = [r["value_ev"] for r in rows]
    clock = [r["value_ev_position"] for r in rows]
    res = [a - b for a, b in zip(ev, clock)]
    return st.median(ev), st.median(clock), st.median(res), sum(r > 0 for r in res) / len(res)

BASE, FLAT3 = "ppo-qwen3-17b-v3-dense-s0", "ppo-qwen3-17b-v3-flat3-s0"
b = {r["step"]: r for r in evals(BASE)}
print(f"  {'step':>5} | {'flat x3':^32} | {'08 baseline':^32}")
for r in evals(FLAT3):
    o = b.get(r["step"], {})
    print(f"  {r['step']:>5} | exact {r['exact_match']:.3f} flags {r['flags_acc']:.4f} "
          f"cause {r['cause_acc']:.3f} | exact {o.get('exact_match', 0):.3f} "
          f"flags {o.get('flags_acc', 0):.4f} cause {o.get('cause_acc', 0):.3f}")

print()
print("  the three gates this run had to hold, checked every evaluation point")
acc = [r["numeric_acc"] for r in evals(FLAT3)]
print(f"    arithmetic   min numeric_acc {min(acc):.3f}")
low = [r["flags_acc"] for r in evals(FLAT3) if r["step"] and r["flags_acc"] < 0.665]
print(f"    no regress   points below the 0.665 water line: {len(low)}")
for run, lab in ((FLAT3, "flat x3   "), (BASE, "08 baseline")):
    ev, clock, res, ahead = critic(run)
    print(f"    critic  {lab} value_ev {ev:+.3f}  clock {clock:+.3f}  "
          f"residual {res:+.4f}  ahead {ahead:.0%}")

## Four hundred steps, and a replicate that nearly changed the verdict

`train()` has no resume, so extending meant re-running with `--steps 400` and the same seed.
That was expected to reproduce the first 200 steps exactly. **It did not:**

| step | 0 | 1 | 2 | **3** |
| --- | --- | --- | --- | --- |
| first run, `reward_mean` | 0.667500 | 0.700000 | 0.665000 | **0.635000** |
| second run, same command | 0.667500 | 0.700000 | 0.665000 | **0.640000** |

The seed is the same, `rng.choice` draws the same batches, and there is no schedule tied to
`cfg.steps`. The cause is that bf16 matmuls on this card are not bitwise deterministic, and at
`temperature = 1.0` one flipped logit flips one sampled token, after which the trajectories
separate completely.

**So "same seed" does not mean "same trajectory" in this series**, and every single-seed
comparison in it is weaker than it reads — including `08`'s own `lam` ablation. What it buys
here is that the 400-step run is a genuine *replicate*, which is what `08` was missing.

And the replicate is the reason this notebook does not report 0.255. At matched steps it was
far behind — `flags_acc` 0.545 against 0.670 at step 25, still 0.748 against 0.860 at 175 —
and then it went past:

| step | 100 | 175 | 200 | 225 | 300 | **400** |
| --- | --- | --- | --- | --- | --- | --- |
| `exact_match` | 0.120 | 0.110 | 0.275 | 0.360 | 0.370 | **0.405** |
| `flags_acc` | 0.670 | 0.748 | 0.863 | 0.878 | 0.867 | **0.927** |
| `cause_acc` | 0.350 | 0.430 | 0.420 | 0.555 | 0.650 | **0.675** |

Both runs sit on the 0.665 never-says-`flat` floor for over a hundred steps and then leave it.
The effect reproduces **in direction and in shape, with the timing varying by about 50 steps** —
which is the same sentence `05` had to write about the entropy bonus, and for the same reason.

### The final policy, measured

| | `08` trained | **this run, step 400** |
| --- | --- | --- |
| `flat` recall | **0/184 = 0.000** | **181/184 = 0.984** |
| non-flat recall | 0.959 | 0.901 |
| `sharp_up` recall | 0.895 | 0.663 |
| all three flags right | 43/200 = 0.215 | **156/200 = 0.780** |
| `cause_acc` | 0.435 | **0.675** |
| `exact_match` | 0.145 | **0.405** |
| causes emitted | 4/7 | **5/7** |

`flat` goes from never to 98.4%. It is not free — non-flat falls 0.959 → 0.901 and `sharp_up`
carries most of that — but all-three-flags nearly quadruples, and that is the quantity the
task is actually gated on.

Two numbers deserve to be called out. **`cause_acc` 0.675 is the highest in the series**, past
the 0.525 that `ppo-qwen3-17b-probe-ent-s0` reached at step 100 — and that run's `exact_match`
was 0.000 and its `numeric_acc` 0.060, which is exactly the degenerate shape `06`'s coverage
gate exists to catch. And **coverage is 5/7 causes**, wider than any trained run in `runs/`
and equal to the frozen policy, so this is not `07`'s label collapse wearing a better score.

### The gates, over all seventeen evaluation points

| | |
| --- | --- |
| the arithmetic | `numeric_acc` **1.000**, minimum over all 17 |
| no regression | `flags_acc` ends at 0.927 against the 0.665 water line |
| the critic | residual over a clock **+0.0456**, ahead on **96%** of steps — the best in the series, against `08`'s +0.0319 |

The critic figure matters most here, because it is the one that could have been quietly traded
away to buy the task result. It went up.

In [ ]:
R400 = "ppo-qwen3-17b-v3-flat3-400-s0"

print("the replicate, and the first run at matched steps")
a = {r["step"]: r for r in evals(R400)}
b = {r["step"]: r for r in evals(FLAT3)}
print(f"  {'step':>5}{'exact':>9}{'flags':>9}{'cause':>9}   |{'first run exact':>17}{'flags':>9}")
for s in sorted(a):
    o = b.get(s)
    tail = f"{o['exact_match']:>17.3f}{o['flags_acc']:>9.4f}" if o else ""
    print(f"  {s:>5}{a[s]['exact_match']:>9.3f}{a[s]['flags_acc']:>9.4f}"
          f"{a[s]['cause_acc']:>9.3f}   |{tail}")

print()
print("  gates over all %d evaluation points" % len(a))
print(f"    arithmetic   min numeric_acc {min(r['numeric_acc'] for r in a.values()):.3f}")
print(f"    flags        final {a[max(a)]['flags_acc']:.4f}  against the 0.665 water line")
ev, clock, res, ahead = critic(R400)
print(f"    critic       value_ev {ev:+.3f}  clock {clock:+.3f}  residual {res:+.4f}  ahead {ahead:.0%}")

print()
print("  the two runs are NOT bitwise reproducible despite an identical command:")
for run in (FLAT3, R400):
    rows = [json.loads(l) for l in open(HERE / "runs" / run / "metrics.jsonl")][:4]
    print(f"    {run[-22:]:>24}  " + "  ".join(f"{r['reward_mean']:.6f}" for r in rows))
print("    (identical at steps 0-2, separated at step 3: bf16 matmuls are not")
print("     deterministic, and at temperature 1.0 one flipped token is enough)")

## The bottleneck moved again, and it is now exactly locatable

`07` and `08` both measured `cause_given_flags = 1.000` — three correct flags determine the
root cause, and the lookup was perfect. It is now **0.622**.

That was derivable before it was measured, from counting alone: `flags_acc` 0.877 leaves 74
wrong slots in 600, which can touch at most 74 of the 200 cases, so at least 126 cases had all
three flags right — and `cause_acc` was 0.420, *below* that floor. A lookup that was still
perfect could not produce that number.

So the chain now reads:

```
arithmetic   1.000      the harness computes it            (08)
flags        0.927      all three right on 156/200          (09)
lookup       0.622      <- everything left is here
cause        0.675
exact        0.405
```

156 cases arrive at the table with the right key and 97 of them come out with the right row.
`predicted_cause_hist` says where it goes: `compaction` is emitted **93** times against about
29 true. The policy over-produces one cause at the lookup step, which is `07`'s label collapse
again — a third time, one layer further down, and this time with the flags underneath it
correct so the failure is unambiguous.

If the lookup returned to 1.000, `cause_acc` would be 0.780 rather than 0.675, and
`exact_match` would follow. That is the `10`.

**And nothing here had converged.** `flags_acc` was 0.883 → 0.900 → 0.927 over the last three
evaluation points and `exact_match` 0.405 was still the highest recorded. 400 steps is where
the run stopped, not where it finished.

In [ ]:
# The final policy, from the greedy pass stored in runs/paired/flat3_400.json.
final = json.load(open(HERE / "runs" / "paired" / "flat3_400.json"))["overall"]
base = json.load(open(HERE / "runs" / "paired" / "ablate_ent.json"))["overall"]

print("  where the reward now comes from, and where it stops")
for label, value in (
    ("arithmetic      ", final["numeric_acc"]),
    ("flags (mean)    ", final["flags_acc"]),
    ("lookup, given correct flags", final["cause_given_flags"]),
    ("cause           ", final["cause_acc"]),
    ("action          ", final["action_acc"]),
    ("exact_match     ", final["exact_match"]),
):
    print(f"    {label:28} {value:.3f}")
print()
print(f"  causes emitted: {final['predicted_cause_hist']}")
print(f"  -> {len(final['predicted_cause_hist'])}/7, against 4/7 for 08's policy and 5/7 frozen")
print()
print("  for scale, the best cause_acc this series had before v3:")
print("    ppo-qwen3-17b-probe-ent-s0 step 100   cause 0.525   exact 0.000   numeric 0.060")
print(f"    this run                              cause {final['cause_acc']:.3f}   "
      f"exact {final['exact_match']:.3f}   numeric {final['numeric_acc']:.3f}")

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(HERE / "runs" / "flat_credit_result.png")))